# 02. Training and evaluation: SageMaker XGBoost training job and Batch Transform

1. **Baseline (BP-501):** the logistic regression bar the model has to beat (`reports/baseline_metrics.json`).
2. **Training job (BP-503):** built-in XGBoost on `ml.m5.large`, reading the `train/` and `validation/` channels from S3.
3. **Batch Transform (BP-504):** score the validation and test feature files offline.
4. **Evaluation (BP-505):** choose the threshold on validation, evaluate once on test, compare with the baseline.

Run notebook 01 first: it uploads the data and saves the bucket and role to `data/processed/sagemaker_state.json`.

In [ ]:
# Setup: repository root, config, lab connection and the state saved by earlier notebooks.
from pathlib import Path

import pandas as pd
import sagemaker

from bankpulse_core.aws import lab_boto_session, read_state, s3_uri, update_state
from bankpulse_core.config import load_config

ROOT = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
config = load_config(ROOT / 'config' / 'data_prep.yaml')
STATE = Path(config['output']['processed_dir']) / 'sagemaker_state.json'
state = read_state(STATE)

boto_session = lab_boto_session()                       # bankpulse-lab profile; no keys in code
sm_session = sagemaker.Session(boto_session=boto_session)
sm_client = boto_session.client('sagemaker')
bucket, prefix, role = state['bucket'], state['prefix'], state['role']

from bankpulse_core.evaluation import choose_threshold, compare, evaluate, parse_batch_output
from bankpulse_core.io import read_csv, read_json, upload_file, write_json

assert 's3' in state, 'Run notebook 01 first: it uploads the data lake and saves its locations'
print('bucket:', bucket, '| prefix:', prefix)

## BP-501. Baseline to beat

Logistic regression on the same 68 features, from `scripts/baseline.py`. The test period subscribes far more often (38% vs 6% in training), so absolute numbers differ a lot between validation and test. The fair comparison is model vs baseline **on the same split**.

In [ ]:
baseline = read_json(ROOT / 'reports' / 'baseline_metrics.json')
metrics_shown = ['positive_rate', 'pr_auc', 'roc_auc', 'precision', 'recall', 'f1', 'lift_top_decile']
display(pd.DataFrame({split: {m: baseline[split][m] for m in metrics_shown} for split in ('validation', 'test')}))

## BP-503. XGBoost training job

**SageMaker pieces:**
- **Image URI:** the container with AWS's built-in XGBoost (version 1.7-1), so no training code is uploaded.
- **Estimator:** describes the job: image, role, instance type, where to write the model, hyperparameters.
- **Channels:** named S3 inputs. Built-in XGBoost reads `train` and `validation`, as CSV with the label in the first column and no header (the format `pipeline.run()` writes).
- **`fit()`:** starts the job. SageMaker starts an `ml.m5.large`, copies the channels in, trains, writes `model.tar.gz` to S3, and shuts the instance down. You pay only for the training time.

**Hyperparameters:**
- `objective=binary:logistic` gives probabilities.
- `eval_metric=aucpr` reports PR-AUC on validation every round.
- `early_stopping_rounds=30` stops when validation PR-AUC hasn't improved for 30 rounds, which limits overfitting.
- The tree settings (`max_depth`, `eta`, `min_child_weight`, `colsample_bytree`) were chosen by `scripts/tune_xgboost.py`, a grid search scored on **validation PR-AUC only**, so the test split stays untouched.

**Why the economic indicators are not inputs:** they describe the *period* of the call (the Euribor rate, employment), not the customer. With a time-ordered split, every later period has values never seen in training, so trees use them as a clock and fail on new data. Validation PR-AUC was 0.177 with them and 0.229 without. See `config/data_prep.yaml`.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# Step 1: the built-in XGBoost container for this region.
image_uri = sagemaker.image_uris.retrieve('xgboost', region=sm_session.boto_region_name, version='1.7-1')

# Step 2: hyperparameters. The tree settings come from scripts/tune_xgboost.py, which chose them
# on validation PR-AUC only (local XGBoost 1.7.6 = this container's version).
tuning = read_json(ROOT / 'reports' / 'xgboost_tuning.json')
best = tuning['best']
hyperparameters = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'num_round': tuning['num_round'],
    'early_stopping_rounds': tuning['early_stopping_rounds'],
    'max_depth': best['max_depth'],
    'eta': best['eta'],
    'min_child_weight': best['min_child_weight'],
    'subsample': tuning['fixed']['subsample'],
    'colsample_bytree': best['colsample_bytree'],
}
print('tuned on validation:', {k: best[k] for k in ('max_depth', 'eta', 'min_child_weight', 'colsample_bytree')},
      '-> validation PR-AUC', best['validation_aucpr'])

# Step 3: the job definition. output_path is where SageMaker writes model.tar.gz.
estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=s3_uri(bucket, prefix, 'model-artifacts'),
    base_job_name='bankpulse-xgboost',
    hyperparameters=hyperparameters,
    max_run=3600,                       # stop after an hour at most
    disable_profiler=True,              # no profiler / debugger files next to the model
    debugger_hook_config=False,
    sagemaker_session=sm_session,
)

# Step 4: the two input channels, each a folder in the data lake holding one CSV file.
channels = {
    'train': TrainingInput(s3_uri(bucket, prefix, 'train/'), content_type='text/csv'),
    'validation': TrainingInput(s3_uri(bucket, prefix, 'validation/'), content_type='text/csv'),
}

# Step 5: start the job and wait for it (about 4-6 minutes, mostly instance start-up).
estimator.fit(channels, logs='None')
print('training job:', estimator.latest_training_job.name)

In [ ]:
# What the job reports: status, run time, and the final train / validation PR-AUC.
job = sm_client.describe_training_job(TrainingJobName=estimator.latest_training_job.name)
display(pd.Series({
    'status': job['TrainingJobStatus'],
    'instance': job['ResourceConfig']['InstanceType'],
    'billable_seconds': job.get('BillableTimeInSeconds'),
    **{m['MetricName']: round(m['Value'], 4) for m in job.get('FinalMetricDataList', [])},
    'model_artifact': job['ModelArtifacts']['S3ModelArtifacts'].split(f'{bucket}/', 1)[1],
}).to_frame('training job'))

state = update_state(STATE, training_job=job['TrainingJobName'], model_data=job['ModelArtifacts']['S3ModelArtifacts'],
             image_uri=image_uri, hyperparameters={k: str(v) for k, v in hyperparameters.items()})

## BP-504. Batch Transform on validation and test

Batch Transform scores a whole file offline. SageMaker starts an instance with the trained model, splits the input file by line, sends rows in mini-batches (`MultiRecord`), and writes one probability per line to `<input file>.out` in S3. There's no endpoint to manage, and nothing keeps running afterwards.

The two jobs run **in parallel** to save time.

In [ ]:
# Step 1: one transformer per split, from the trained estimator (this registers a SageMaker Model).
output_path = s3_uri(bucket, prefix, 'batch-predictions')
transformers = {
    split: estimator.transformer(instance_count=1, instance_type='ml.m5.large', strategy='MultiRecord',
                                 assemble_with='Line', accept='text/csv', output_path=output_path)
    for split in ('validation', 'test')
}

# Step 2: start both jobs without waiting; each reads a feature-only file and splits it by line.
for split, transformer in transformers.items():
    transformer.transform(data=state['s3'][f'{split}_features'], content_type='text/csv', split_type='Line',
                          wait=False)

# Step 3: wait for both (about 5-7 minutes).
for split, transformer in transformers.items():
    transformer.wait(logs=False)
    print(split, 'transform job:', transformer.latest_transform_job.name)

predictions = {split: s3_uri(bucket, prefix, 'batch-predictions', f'{split}_features.csv.out')
               for split in transformers}
state = update_state(STATE, batch_predictions=predictions,
             transform_jobs={s: t.latest_transform_job.name for s, t in transformers.items()})
display(pd.Series(predictions).to_frame('predictions'))

## BP-505. Threshold, test evaluation and comparison

- **Threshold:** the cut-off with the best F1, chosen on **validation** only.
- **Test:** evaluated **once** with that threshold, so the test result isn't tuned to the test set.
- **Headline metric:** PR-AUC, because only 11% of customers subscribe and accuracy would be misleading.

In [ ]:
# Step 1: scores from the .out files and labels from the data lake, both read from S3.
scores = {split: parse_batch_output(uri) for split, uri in predictions.items()}
labels = {split: read_csv(state['s3'][f'{split}_labels'], header=None)[0].to_numpy() for split in predictions}

# Step 2: threshold on validation; one evaluation per split with that same threshold.
threshold = choose_threshold(labels['validation'], scores['validation'])
xgboost_metrics = {
    'model': 'xgboost',
    'training_job': state['training_job'],
    'validation': evaluate(labels['validation'], scores['validation'], threshold),
    'test': evaluate(labels['test'], scores['test'], threshold),
}

# Step 3: side by side with the baseline, per split.
table = pd.DataFrame({
    f'{model} {split}': {m: metrics[split][m] for m in metrics_shown}
    for split in ('validation', 'test')
    for model, metrics in (('baseline', baseline), ('xgboost', xgboost_metrics))
})
display(table)
print('threshold chosen on validation:', round(threshold, 4))

In [ ]:
# Step 4: which model wins on test, metric by metric (PR-AUC decides overall).
comparison = compare(xgboost_metrics['test'], baseline['test'])
display(pd.DataFrame({m: v for m, v in comparison.items() if isinstance(v, dict)}).T)
print('XGBoost beats the baseline on test PR-AUC:', comparison['candidate_beats_baseline'])

In [ ]:
# Step 5: precision-recall curves and the test confusion matrix.
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for split in ('validation', 'test'):
    PrecisionRecallDisplay.from_predictions(labels[split], scores[split], ax=axes[0],
                                            name=f'XGBoost {split} (PR-AUC {xgboost_metrics[split]["pr_auc"]})')
axes[0].set_title('Precision-recall curves')
ConfusionMatrixDisplay.from_predictions(labels['test'], (scores['test'] >= threshold).astype(int), ax=axes[1],
                                        display_labels=['no', 'yes'], colorbar=False)
axes[1].set_title(f'Test confusion matrix (threshold {threshold:.3f})')
plt.tight_layout()

In [ ]:
# Step 6: save the results locally (committed: no account details) and in the data lake.
metrics_path = write_json(xgboost_metrics | {'threshold': round(threshold, 6)}, ROOT / 'reports' / 'xgboost_metrics.json')
comparison_path = write_json(comparison, ROOT / 'reports' / 'model_comparison.json')
metrics_uri = upload_file(metrics_path, s3_uri(bucket, prefix, 'reports', 'xgboost_metrics.json'))
upload_file(comparison_path, s3_uri(bucket, prefix, 'reports', 'model_comparison.json'))
state = update_state(STATE, threshold=threshold, metrics_uri=metrics_uri)
print('saved', metrics_path.name, 'and', comparison_path.name, '(local and S3)')